# ITAM Tuesday 27 October 2026: where did Lucy's phone number disappear?

This notebook follows one made-up delivery note through a document pipeline, from the page to the field an assistant would use. Every result a class card shows comes from a cell of this notebook, run from top to bottom. Each result is labeled:

- **LIVE**: computed by the cell you are looking at.
- **RECORDED**: read from a saved file of an earlier run. The two cells that talk to a hosted model (Jev and the hosted session) replay the saved answers and say so, because they need the class keys.

Evidence levels used in the text: OBSERVED (what a run shows), HYPOTHESIS (a possible cause, not established), INTERVENTION (a change we made, and what came back).

The delivery notes, the phone number "55 0100" and the companies are made up for this class. "55 0100" is a short classroom identifier, not a number anyone can dial.

**To run it in Google Colab:** open the notebook, run all cells (the first cell installs the packages and takes several minutes). **First cell:** it prints the Python version, the platform and every package version, so a reader can see what produced the outputs below. The direct packages are pinned in that cell; the versions of what they depend on are chosen by pip when it installs. The class render of these outputs used one exact set of 198 packages for macOS on Apple silicon (not a set built for Colab), and a Colab install may differ in the third decimal of some scores.

In [ ]:
# 0. Setup: pinned installs, the inputs, and a record of the environment
import os, sys, subprocess, platform, shutil, hashlib, json, urllib.request, zipfile, io, time
from pathlib import Path

# Where the inputs live. In Colab this is the public repository of the class; a local mirror can be named in ITAM_BASE.
BASE = os.environ.get("ITAM_BASE", "https://raw.githubusercontent.com/matorclawson/profrodai-resources/cf6d56383876a355d632ed69d2f4ec7a6ee7e7f9/courses/zeo-itam-autumn-2026/labs/09-document-processing-2026-10-27/bundle")
PINS = ["docling==2.142.1", "paddlepaddle==3.3.1", "paddleocr==3.7.0",
        "unstructured==0.27.35", "typesafe-sdk==0.7.5", "pillow==13.0.0"]
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], capture_output=True, text=True)
print("pip install exit code:", r.returncode, f"({time.time()-t0:.0f} s)")
if r.returncode: print(r.stderr[-1500:]); raise SystemExit("install failed")

# Tesseract is a program, not a Python package. Colab installs it with apt; here we use one that is already present.
if shutil.which("tesseract") is None:
    subprocess.run(["apt-get", "-qq", "install", "-y", "tesseract-ocr"], check=True)

WORK = Path(os.environ.get("ITAM_WORK", "work")).resolve(); WORK.mkdir(parents=True, exist_ok=True); os.chdir(WORK)
sys.path.insert(0, str(WORK))
FILES = json.loads(urllib.request.urlopen(BASE + "/MANIFEST.json").read())      # {relative path: sha256}
for rel, want in FILES.items():
    data = urllib.request.urlopen(BASE + "/" + rel).read()
    got = hashlib.sha256(data).hexdigest()
    assert got == want, f"{rel}: sha256 differs from the manifest"
    dest = WORK / rel; dest.parent.mkdir(parents=True, exist_ok=True); dest.write_bytes(data)
for f in (WORK / "code").glob("*.py"): shutil.copy(f, WORK / f.name)            # accept_phone.py, jevrec.py, agentsapi.py next to the cells
os.environ["FIX"] = str(WORK / "fixtures"); os.environ["OUT2"] = str(WORK / "research")

import importlib.metadata as md_
def v(p):
    try: return md_.version(p)
    except Exception: return "not installed"
print("inputs:", len(FILES), "files fetched from", BASE.split("/")[0] + "//" + BASE.split("/")[2], "and checked against the manifest")
print("python    :", sys.version.split()[0], "|", platform.platform(), "(Colab's Python version may differ from this one)")
print("tesseract :", subprocess.run(["tesseract", "--version"], capture_output=True, text=True).stdout.splitlines()[0])
for p in ["docling", "docling-core", "rapidocr", "torch", "paddlepaddle", "paddleocr", "unstructured", "typesafe-sdk", "pillow", "numpy"]:
    print(f"{p:14}", v(p))
print("run at    :", time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
if os.environ.get("ITAM_RENDER_NOTE"): print("render note:", os.environ["ITAM_RENDER_NOTE"])

## Part 1. Follow the field

Lucy wants to call Fresas del Norte to confirm Saturday's
strawberries. She needs the supplier's phone number from the
delivery note. This note is made up for the class and its
true number is 55 0100, so we can see exactly where a number
is lost.

In [ ]:
# 1.1 the fixtures: one page, six versions
import hashlib, os
from pathlib import Path
FIX = Path(os.environ["FIX"])
TRUTH = "55 0100"
for f in sorted(FIX.glob("fn-*.*")):
    if f.suffix in {".txt"}: continue
    digest = hashlib.sha256(f.read_bytes()).hexdigest()[:12]
    size = f.stat().st_size
    print(f"{f.name:24} {size:7} bytes  sha256 {digest}")
print("true phone:", TRUTH)

In [ ]:
# 1.2 read the faint-patch page with Docling and look at the scores
import logging, warnings
warnings.filterwarnings("ignore"); logging.disable(logging.CRITICAL)
from docling.document_converter import DocumentConverter
conv = DocumentConverter()
def read(name):
    r = conv.convert(str(FIX / name))
    return r, r.pages[0], r.document.export_to_markdown()
r_spot, p_spot, md_spot = read("fn-spot.jpg")
sc = r_spot.confidence.pages[1]
print("page ocr_score   :", round(float(sc.ocr_score), 3))
print("page layout_score:", round(float(sc.layout_score), 3))
tel = [l for l in md_spot.splitlines() if "Tel" in l]
print("phone line read  :", tel)

In [ ]:
# 1.3 follow the field: the layout box around the phone line
r_mild, p_mild, md_mild = read("fn-mild.jpg")
def phone_box(page):
    for c in page.predictions.layout.clusters:
        has_tel = any("Tel" in x.text for x in c.cells)
        if str(c.label.value) == "text" and has_tel:
            b = c.bbox
            return {"box": [round(b.l), round(b.t), round(b.r),
                            round(b.b)],
                    "confidence": round(float(c.confidence), 3),
                    "cells": [x.text for x in c.cells]}
print("mild scan        :", phone_box(p_mild))
print("faint-patch scan :", phone_box(p_spot))
print("the reported faint-patch box ends earlier and holds only Tel. 55: "
      "consistent with the digits never reaching the reader (a hypothesis); "
      "Part 4 tests a repair, not the cause")

## Part 2. Choose a reading route, and three failures

Before a page is read, the assistant has to choose a route.
Three files arrive for the same note: one born digital, one
with a text layer that may be wrong, one a scan.

In [ ]:
# 2.1 how many characters are already in each file?
import logging, os, warnings
warnings.filterwarnings("ignore"); logging.disable(logging.CRITICAL)
from pathlib import Path
from PIL import Image
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter
from docling.document_converter import PdfFormatOption
FIX = Path(os.environ["FIX"])
scan = Image.open(FIX / "fn-mild.jpg").convert("RGB")
scan.save("fn-mild-scan.pdf", resolution=150)   # scan as a PDF
opts = PdfPipelineOptions(do_ocr=False, do_table_structure=False)
text_only = DocumentConverter(format_options={
    InputFormat.PDF: PdfFormatOption(pipeline_options=opts)})
FILES = [("born-digital", FIX / "fn-born-digital.pdf"),
         ("embedded-wrong", FIX / "fn-embedded-wrong.pdf"),
         ("scan", Path("fn-mild-scan.pdf"))]
for name, path in FILES:
    doc = text_only.convert(str(path)).document
    n = len(doc.export_to_text().strip())
    print(f"{name:15} {n:4} characters in the file")

In [ ]:
# 2.2 what a text-only read returns, and its scores
def tel_text(text):
    line = [l for l in text.splitlines() if "Tel" in l][0]
    return line[line.index("Tel"):][:12]
LAYER = {}
for name in ["fn-born-digital.pdf", "fn-embedded-wrong.pdf"]:
    r = text_only.convert(str(FIX / name))
    s = r.confidence.pages[1]
    LAYER[name] = tel_text(r.document.export_to_text())
    score = float(s.parse_score)
    print(f"{name:22} parse_score {score:.1f}   {LAYER[name]}")
print("both score 1.0; one number is right and one is wrong")

In [ ]:
# 2.2b the same file converted with Docling's default settings
shown = Image.open(FIX / "fn-clean.png").convert("L")
# the phone line of the full size note (x 80 to 640, y 262 to 329)
shown.crop((80, 262, 640, 329)).resize((1380, 165)).save(
    "visible-line.png")
full = DocumentConverter()
r = full.convert(str(FIX / "fn-embedded-wrong.pdf"))
default = tel_text(r.document.export_to_markdown())
layer = LAYER["fn-embedded-wrong.pdf"]
ocr = float(r.confidence.pages[1].ocr_score)
print("text only read     :", layer)
print("default read       :", default)
print("default read ocr_score (nan means no OCR):", round(ocr, 3))
print("the two agree:", layer == default)

In [ ]:
# 2.3 the heavy scan: every reader gets garbage, no phone line
import subprocess
heavy = str(FIX / "fn-heavy.jpg")
d = DocumentConverter().convert(heavy)
md = d.document.export_to_markdown()
score = d.confidence.pages[1].ocr_score
print("docling ocr_score:", round(float(score), 3))
print("docling first lines:", md.splitlines()[:4])
cmd = ["tesseract", heavy, "stdout", "--psm", "4", "-l", "eng"]
t = subprocess.run(cmd, capture_output=True, text=True).stdout
first = [l for l in t.splitlines() if l.strip()][:3]
print("tesseract first lines:", first)
by_docling = any("Tel" in l for l in md.splitlines())
print("phone line found by docling:", by_docling,
      " by tesseract:", "Tel" in t)
print("the system stops here and asks for a better source")

## Part 3. The agent in control (the hosted answers are RECORDED)

Now the assistant runs the work. A hosted session reads the
faint-patch note with four readers. Jev RECOMMENDS at each
step, and our code APPLIES the policy. "55 0100" is a short
fictional classroom identifier, not a usable phone number.
What leaves this machine in the recorded calls: synthetic
extracted TEXT strings only (the phone lines the readers
returned and file facts such as character counts). No images.
What is real and what is narrated: Jev's route recommendation
is recorded and printed, and it dispatches no reader. The
hosted session calls our parse_note function, which returns
the recorded strings of the research runs, not a live parser.
The policy check in accept_phone.py is deterministic code on
those strings.
NEEDS THE CLASS SDKs AND KEYS (agentsapi.py, jevrec.py,
keys.py): record with RECORD=1, replay without.

In [ ]:
# 3.1 a typed question for Jev: how should this file be read?
from typesafe_sdk import Choice
READS = {
    "text_layer": ("The file already holds the text. Use it as "
                   "it is."),
    "ocr": ("The page is a picture. Read it with "
            "OCR."),
    "check_text_layer": (
        "The file holds text, but it may be wrong. Read the page "
        "picture too and compare.")}
QUESTION = {"read": Choice(
    instructions=("A delivery note page has arrived for Lucy's "
                  "shop. Pick the one way to read it that fits "
                  "what we know about the file."),
    criteria=READS)}

In [ ]:
# 3.2 what we know about each file, one call to Jev each
from jevrec import ask
FILES = {
    "born-digital": {"characters_in_file": 111,
                     "document_parse_score": 1.0,
                     "identifier_has_valid_shape": True},
    "embedded-wrong": {"characters_in_file": 202,
                       "document_parse_score": 1.0,
                       "identifier_has_valid_shape": False},
    "scan": {"characters_in_file": 0}}
for name, state in FILES.items():
    p = ask(f"s03-route-{name}", state, QUESTION)["read"]
    top = max(p, key=p.get)
    print(f"{name:15} Jev recommends {top:16} {p[top]:.2f}")

In [ ]:
# 3.3 the hosted session reports a candidate with its evidence
from agentsapi import lucy_session
TOOLS = ["tesseract", "docling", "granite_docling", "mineru_flash"]
PARSE_TOOL = {
    "type": "function", "name": "parse_note",
    "description": ("Return the phone line a named reader returned "
                    "for one delivery note (extracted text only)."),
    "parameters": {"type": "object", "properties": {
        "note": {"type": "string"},
        "tool": {"type": "string", "enum": TOOLS}},
        "required": ["note", "tool"]}}
READS_BY_TOOL = {"tesseract": "Tel.:S5", "docling": "Tel. 55",
                 "granite_docling": "Tel. 55",
                 "mineru_flash": "Tel.55 0100"}   # raw, from t4
def parse_note(note, tool): return READS_BY_TOOL[tool]
agent = {"model": "gpt-6-astra",
         "instructions": (
    "You are Lucy's ingestion operator. Call parse_note once per "
    "tool for the note, then report the phone identifier as a "
    "CANDIDATE: the value if a tool returned it whole, which "
    "tools agreed, and which tools returned only part of it. Do "
    "not guess missing digits. You do not accept or remember "
    "anything; you only report. You have no file tool: do not "
    "say you saved anything."),
         "tools": [PARSE_TOOL]}
run = lucy_session("s03-agent-1", agent,
                   "Report the phone candidate for fn-spot.",
                   {"parse_note": parse_note})
print(run.final_output)

In [ ]:
# 3.4 Jev recommends; our code applies the written policy
from accept_phone import judge
DECIDE = {"recommendation": Choice(
    instructions=("Lucy must contact a supplier. Four readers "
                  "returned their phone line for one delivery "
                  "note. Recommend what the assistant should do "
                  "with the identifier."),
    criteria={
        "accept": ("One complete identifier, returned by more "
                   "than one reader."),
        "re_read": ("A reader returned only part of the "
                    "identifier. Read the line again."),
        "ask_lucy": ("No reader returned the identifier. Ask "
                     "Lucy for a better photo.")})}
state = {"reads": READS_BY_TOOL}
p = ask("s03-decide-fn-spot", state, DECIDE)["recommendation"]
top = max(p, key=p.get)
print("Jev recommends:", top, f"{p[top]:.2f}")
# asserted: the layout box of the research run (t1 cell03)
REGIONS = {"docling": {"page": 1, "box": [77, 209, 210, 240]}}
j = judge("Fresas del Norte", READS_BY_TOOL, REGIONS,
          page_readable=True)
print("raw reader output  :", j["raw"])
norm = {t: v["value"] for t, v in j["normalised"].items()}
print("normalised values  :", norm)
print("policy result      :", j["state"], "|", j["action"])
print("candidate          :", j["candidate"],
      "| candidate evidence:", j.get("candidate_evidence"))
print("accepted fact      :", j["accepted_fact"])
written = "yes" if j["accepted_fact"] else "no"
print("application writes accepted facts only:", written)

## Part 4. Choose a repair, and decide

Back to the faint patch page. We ask what changes the result:
the region we read, the size we read it at, or both. Then our
code applies a written policy to what came back. "55 0100" is
a short fictional classroom identifier, not a usable phone
number. In this fixture a complete identifier is two digits,
a space, and four digits.

In [ ]:
# 4.1 OBSERVED: the same page read several ways, same reader
import os, subprocess
from pathlib import Path
from PIL import Image, ImageOps
FIX = Path(os.environ["FIX"])
def tess(img, psm):
    cmd = ["tesseract", img, "stdout", "--psm", str(psm),
           "-l", "eng"]
    return subprocess.run(cmd, capture_output=True,
                          text=True).stdout.strip()
def tel_lines(text):
    return [l for l in text.splitlines() if "Tel" in l or "55" in l]
page = Image.open(FIX / "fn-spot.jpg").convert("L")
page.save("page-gray.png")
box = (60, 195, 520, 250)   # the phone line, wider than the digits
page.crop(box).save("region-only.png")
w, h = page.size
page.resize((w * 3, h * 3), Image.LANCZOS).save("scale-only.png")
line = page.crop(box)
line = line.resize((line.width * 3, line.height * 3), Image.LANCZOS)
line.save("line.png")
JPG = FIX / "fn-spot.jpg"
RUNS = [("whole page, original JPG, mode 4", JPG, 4),
        ("whole page, grayscale PNG, mode 4", "page-gray.png", 4),
        ("line crop, mode 4", "region-only.png", 4),
        ("line crop, mode 7 (one line)", "region-only.png", 7),
        ("whole page, 3 times, mode 4", "scale-only.png", 4),
        ("line crop, 3 times, mode 7", "line.png", 7)]
for label, path, psm in RUNS:
    print(f"{label:36}", tel_lines(tess(str(path), psm)))
ImageOps.autocontrast(line, cutoff=1).save("line-contrast.png")
label = "line crop, 3 times, contrast boost"
print(f"{label:36}", tel_lines(tess("line-contrast.png", 7)))

What we can say from this one page and one reader (Tesseract).
OBSERVED: the faint digits are lost when reading the whole page,
in the original file and in a grayscale copy. HYPOTHESIS: the
region we hand the reader may exclude them. INTERVENTION: a
crop to the phone line recovered the identifier in modes 4 and
7, and enlarging the whole page did not. We do not claim the
cause beyond what these rows show.

In [ ]:
# 4.2 raw reader output and the application's normalisation
from accept_phone import normalise
raw = tess("line.png", 7)
value, rule = normalise(raw)
print("raw reader output   :", repr(raw))
print("normalised value    :", repr(value))
print("rule that produced it:", rule)

**RECORDED, not live.** The page scores and the reader lines in the next cell are read from files saved by an earlier run of the same readers, in an environment with different library versions. A third decimal can therefore differ from the live cells above (for example 0.985 here against 0.986 in Part 1). Part 1 shows the live value; these saved files are the inputs to the policy.

In [ ]:
# 4.3 the written policy applied to what the four readers returned
# Asserted teaching inputs, flagged here: the regions come from the
# layout trace of the research run, and a page counts as readable
# when Docling's page ocr_score is at least 0.80, a threshold
# chosen for the class.
import json
from accept_phone import judge
OUT = Path(os.environ["OUT2"])
SUMMARY = json.loads((OUT / "docling-fn-summary.json").read_text())
def line_of(path):
    for l in Path(path).read_text().splitlines():
        if "Tel" in l: return l[l.index("Tel"):]
    return None
def tel_region(case):
    f = OUT / f"trace-{case}.json"
    if not f.exists(): return {}
    for c in json.loads(f.read_text())["layout_clusters"]:
        has_tel = any("Tel" in x for x in c["cells"])
        if c["label"] == "text" and has_tel:
            box = [round(v) for v in c["bbox"]]
            return {"docling": {"page": 1, "box": box}}
    return {}
def reads_for(case):
    return {
        "tesseract": line_of(OUT / f"tess-{case}.txt"),
        "docling": line_of(OUT / f"docling-{case}.md"),
        "granite": line_of(OUT / f"graniteVLM-{case}.md"),
        "mineru_flash": line_of(OUT / f"mineru-flash-{case}.md")}
for case in ["fn-mild", "fn-spot", "fn-heavy"]:
    reads = reads_for(case)
    ocr = SUMMARY[case]["ocr"]
    readable = ocr >= 0.80
    j = judge("Fresas del Norte", reads, tel_region(case),
              page_readable=readable)
    print(case, "page ocr_score", round(ocr, 3),
          "| page readable:", readable)
    print("   state:", j["state"], "|", j["action"])
    print("   raw       :", j["raw"])
    norm = {t: v["value"] for t, v in j["normalised"].items()}
    print("   normalised:", norm)
    print("   candidate :", j["candidate"],
          "| candidate evidence:", j.get("candidate_evidence"))
    print("   memory write:",
          json.dumps({"accepted_fact": j["accepted_fact"]}))

On the faint patch page the assistant keeps the number only as
a candidate. The code that acts on memory reads accepted
facts, and there is none. On the heavy scan we do not know
whether the page has a phone line, so the state is
unreadable, and absence is kept for a page we could read. A
contact recommendation can name what is still needed: a
better photo of the note, or a second reader that sees the
whole line.

## Part 5. A second OCR family: PaddleOCR reads the same notes

So far the readers were Docling, Tesseract and two replayed ones. PaddleOCR is a different tool with its own detection and recognition models. We run it on four of the six versions of the note. OBSERVED results only; one run per file, so this is a look at behavior and not a ranking.

In [ ]:
# 5.1 PaddleOCR on four versions of the note (LIVE)
import warnings, logging, os
warnings.filterwarnings("ignore"); logging.disable(logging.CRITICAL)
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
from paddleocr import PaddleOCR
kw = dict(use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,
          text_detection_model_name="PP-OCRv5_mobile_det", text_recognition_model_name="PP-OCRv5_mobile_rec")
det, rec = os.environ.get("PP_OCR_DET_DIR"), os.environ.get("PP_OCR_REC_DIR")
if det and rec:
    kw.update(text_detection_model_dir=det, text_recognition_model_dir=rec); print("PaddleOCR models: pinned local folders")
else:
    print("PaddleOCR models: PP-OCRv5 mobile, downloaded by PaddleOCR on first use")
ocr = PaddleOCR(**kw)
def paddle_lines(name):
    res = ocr.predict(str(Path(os.environ["FIX"]) / name))
    return [(round(float(s), 3), t) for r in res for t, s in zip(r["rec_texts"], r["rec_scores"])]
PADDLE = {}
for name in ["fn-clean.png", "fn-mild.jpg", "fn-spot.jpg", "fn-heavy.jpg"]:
    PADDLE[name] = paddle_lines(name)
    tel = [l for l in PADDLE[name] if "tel" in l[1].lower()]
    print(f"{name:14} lines read: {len(PADDLE[name]):2} | phone line: {tel}")

In [ ]:
# 5.2 a fifth reader goes through the same written policy (LIVE, on the strings above)
from accept_phone import judge
def tel_of(lines):
    t = [x[1] for x in lines if "tel" in x[1].lower()]
    return t[0] if t else None
reads5 = dict(READS_BY_TOOL); reads5["paddleocr"] = tel_of(PADDLE["fn-spot.jpg"])
j5 = judge("Fresas del Norte", reads5, REGIONS, page_readable=True)
print("raw reader output  :", j5["raw"])
print("normalised values  :", {t: v["value"] for t, v in j5["normalised"].items()})
print("policy result      :", j5["state"], "|", j5["action"])
print("candidate          :", j5["candidate"], "| accepted fact:", j5["accepted_fact"])

## Part 6. Text that is already text: Unstructured

Not every input is a picture. Unstructured splits a document into typed elements. We give it the born-digital text of the note. This base install does not read PDFs or images, so it cannot take the scanned versions. On first use it downloads a small English language model (en_core_web_sm 3.8.0, about 15 MB, from GitHub) and checks it against a fixed hash; in the class render that model was already installed. OBSERVED: what it returned for this one small file.

In [ ]:
# 6.1 Unstructured partitions the born-digital text (LIVE)
os.environ.setdefault("UNSTRUCTURED_DISABLE_NOTICE", "1")
from unstructured.partition.text import partition_text
els = partition_text(filename=str(Path(os.environ["FIX"]) / "fn-born-digital.txt"))
print(len(els), "element(s)")
for e in els: print(type(e).__name__, "|", str(e))

## What this notebook shows

A reader score describes a reading, not the field. The faint-patch note scored high and still gave a short phone line; the box and the crop experiments suggest where the digits were lost, and a written policy kept the candidate out of memory. A second OCR family and a plain-text partitioner behave differently on the same notes. Every result above was produced, or replayed and labeled, by the cell above it.